eICU-CRD external validation (version 3.0)
Stage 1: extraction and analysis dataset; Stage 2: analyses. Requires credentialed access to eICU-CRD v2.0 on PhysioNet and BigQuery access.
Run cells in order in Google Colab. Patient-level data stay in this notebook;
only the aggregated summaries printed at the end are to be shared.

In [ ]:
# Cell 0: setup
PROJECT_ID = "YOUR_PROJECT_ID"   # <- your Google Cloud project
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
import pandas as pd, numpy as np
client = bigquery.Client(project=PROJECT_ID)
def q(sql):
    return client.query(sql).to_dataframe(create_bqstorage_client=True)


In [ ]:
# Cell 1: SQL building blocks (eligible stays: adult first ICU stay >= 48 h, no ESKD,
# blood gas (pH + HCO3) and lactate on both days, or SBE-lactate pairs within 60 min on both days)
ELIG = r"""
WITH pt AS (
  SELECT p.patientunitstayid, p.uniquepid,
         CASE WHEN p.age = '> 89' THEN 90 ELSE SAFE_CAST(p.age AS INT64) END AS age,
         p.unitdischargeoffset,
         ROW_NUMBER() OVER (PARTITION BY p.uniquepid
           ORDER BY p.hospitaldischargeyear, p.patienthealthsystemstayid, p.unitvisitnumber) AS rn
  FROM `physionet-data.eicu_crd.patient` p
),
s3 AS (
  SELECT patientunitstayid FROM pt
  WHERE rn = 1 AND age >= 18 AND unitdischargeoffset >= 2880
    AND patientunitstayid NOT IN (
      SELECT patientunitstayid FROM `physionet-data.eicu_crd.diagnosis`
      WHERE REGEXP_CONTAINS(IFNULL(icd9code, ''), r'585\.6|N18\.6|V45\.11|Z99\.2')
         OR REGEXP_CONTAINS(LOWER(diagnosisstring), r'esrd|end stage renal|end-stage renal')
      UNION DISTINCT
      SELECT patientunitstayid FROM `physionet-data.eicu_crd.pasthistory`
      WHERE REGEXP_CONTAINS(LOWER(pasthistorypath), r'dialysis'))
),
gl AS (
  SELECT patientunitstayid, labresultoffset AS t, LOWER(labname) AS nm, labresult AS v
  FROM `physionet-data.eicu_crd.lab`
  WHERE LOWER(labname) IN ('ph', 'hco3', 'lactate') AND labresult IS NOT NULL
    AND labresultoffset BETWEEN -360 AND 2880
    AND patientunitstayid IN (SELECT patientunitstayid FROM s3)
),
ph AS (SELECT patientunitstayid, t, AVG(v) v FROM gl WHERE nm = 'ph' AND v BETWEEN 6.5 AND 8.0 GROUP BY 1, 2),
hc AS (SELECT patientunitstayid, t, AVG(v) v FROM gl WHERE nm = 'hco3' AND v BETWEEN 2 AND 60 GROUP BY 1, 2),
gas AS (SELECT patientunitstayid, t, IF(t <= 1440, 1, 2) day FROM ph JOIN hc USING (patientunitstayid, t)),
lac AS (SELECT patientunitstayid, t, IF(t <= 1440, 1, 2) day FROM gl WHERE nm = 'lactate' AND v > 0 AND v < 30),
gf AS (SELECT patientunitstayid, MAX(IF(day = 1, 1, 0)) g1, MAX(IF(day = 2, 1, 0)) g2 FROM gas GROUP BY 1),
lf AS (SELECT patientunitstayid, MAX(IF(day = 1, 1, 0)) l1, MAX(IF(day = 2, 1, 0)) l2 FROM lac GROUP BY 1),
pr AS (
  SELECT g.patientunitstayid, g.day FROM gas g JOIN lac l
    ON g.patientunitstayid = l.patientunitstayid AND ABS(g.t - l.t) <= 60
  GROUP BY 1, 2
),
pf AS (SELECT patientunitstayid, MAX(IF(day = 1, 1, 0)) b1, MAX(IF(day = 2, 1, 0)) b2 FROM pr GROUP BY 1),
elig AS (
  SELECT s3.patientunitstayid,
         IFNULL(gf.g1 * gf.g2 * lf.l1 * lf.l2, 0) AS dm_ok,
         IFNULL(pf.b1 * pf.b2, 0) AS p60_ok
  FROM s3
  LEFT JOIN gf USING (patientunitstayid)
  LEFT JOIN lf USING (patientunitstayid)
  LEFT JOIN pf USING (patientunitstayid)
  WHERE IFNULL(gf.g1 * gf.g2 * lf.l1 * lf.l2, 0) = 1 OR IFNULL(pf.b1 * pf.b2, 0) = 1
)
"""


In [ ]:
# Cell 2: stay-level data
stay = q(ELIG + r"""
SELECT e.patientunitstayid, e.dm_ok, e.p60_ok,
       p.hospitalid,
       CASE WHEN p.age = '> 89' THEN 90 ELSE SAFE_CAST(p.age AS INT64) END AS age,
       p.gender, p.unittype, p.unitadmitsource, p.admissionweight,
       p.unitdischargeoffset, p.unitdischargestatus,
       p.hospitaldischargeoffset, p.hospitaldischargestatus,
       ap.acutephysiologyscore AS aps, ap.apachescore,
       pv.electivesurgery
FROM elig e
JOIN `physionet-data.eicu_crd.patient` p USING (patientunitstayid)
LEFT JOIN (
  SELECT patientunitstayid, MAX(acutephysiologyscore) acutephysiologyscore, MAX(apachescore) apachescore
  FROM `physionet-data.eicu_crd.apachepatientresult`
  WHERE apacheversion = 'IV' AND acutephysiologyscore >= 0
  GROUP BY 1
) ap USING (patientunitstayid)
LEFT JOIN (
  SELECT patientunitstayid, MAX(electivesurgery) electivesurgery
  FROM `physionet-data.eicu_crd.apachepredvar` GROUP BY 1
) pv USING (patientunitstayid)
""")
print("stays:", len(stay))


In [ ]:
# Cell 3: laboratory values (long format)
labs = q(ELIG + r"""
SELECT l.patientunitstayid, l.labresultoffset AS t, LOWER(l.labname) AS nm, l.labresult AS v
FROM `physionet-data.eicu_crd.lab` l
JOIN elig e USING (patientunitstayid)
WHERE LOWER(l.labname) IN ('ph', 'hco3', 'base excess', 'lactate', 'creatinine',
                           'platelets x 1000', 'total bilirubin', 'bun', 'hgb',
                           'chloride', 'sodium', 'albumin')
  AND l.labresult IS NOT NULL
  AND l.labresultoffset BETWEEN -10080 AND 10080
""")
print("lab rows:", len(labs)); print(labs.nm.value_counts())


In [ ]:
# Cell 4: treatments, drugs, fluids, vitals, comorbidities (stay-level)
tx = q(ELIG + r"""
, krt AS (
  SELECT patientunitstayid, MIN(t) AS krt_start FROM (
    SELECT patientunitstayid, treatmentoffset AS t FROM `physionet-data.eicu_crd.treatment`
    WHERE REGEXP_CONTAINS(LOWER(treatmentstring), r'dialysis|crrt|cvvh|hemofiltration|ultrafiltration')
    UNION ALL
    SELECT patientunitstayid, intakeoutputoffset FROM `physionet-data.eicu_crd.intakeoutput`
    WHERE IFNULL(dialysistotal, 0) <> 0
  ) GROUP BY 1
),
vaso AS (
  SELECT DISTINCT patientunitstayid FROM `physionet-data.eicu_crd.infusiondrug`
  WHERE REGEXP_CONTAINS(LOWER(drugname), r'norepinephrine|levophed|epinephrine|vasopressin|phenylephrine|neo-synephrine|neosynephrine|dopamine')
    AND infusionoffset BETWEEN 0 AND 2880
),
mv AS (
  SELECT DISTINCT patientunitstayid FROM `physionet-data.eicu_crd.treatment`
  WHERE REGEXP_CONTAINS(LOWER(treatmentstring), r'mechanical ventilation') AND treatmentoffset <= 2880
  UNION DISTINCT
  SELECT DISTINCT patientunitstayid FROM `physionet-data.eicu_crd.respiratorycare`
  WHERE ventstartoffset IS NOT NULL AND ventstartoffset <> 0 AND ventstartoffset BETWEEN -1440 AND 2880
),
io AS (
  SELECT patientunitstayid,
    SUM(IF(cellpath LIKE '%|Intake (ml)%', cellvaluenumeric, 0)) AS intake_ml,
    SUM(IF(cellpath LIKE '%|Output (ml)%', cellvaluenumeric, 0)) AS output_ml,
    SUM(IF(cellpath LIKE '%|Output (ml)%'
           AND REGEXP_CONTAINS(LOWER(celllabel), r'urine|foley|void|catheter|nephrostomy')
           AND NOT REGEXP_CONTAINS(LOWER(celllabel), r'irrigant|specific|sample'), cellvaluenumeric, 0)) AS urine_ml,
    COUNTIF(cellpath LIKE '%|Output (ml)%'
           AND REGEXP_CONTAINS(LOWER(celllabel), r'urine|foley|void|catheter|nephrostomy')) AS n_urine_rec,
    MAX(IF(REGEXP_CONTAINS(LOWER(celllabel), r'prbc|packed red|red blood cell|\brbc\b') AND cellvaluenumeric > 0, 1, 0)) AS rbc_io
  FROM `physionet-data.eicu_crd.intakeoutput`
  WHERE intakeoutputoffset BETWEEN 0 AND 2880 AND cellvaluenumeric BETWEEN 0 AND 20000
  GROUP BY 1
),
map AS (
  SELECT patientunitstayid, MIN(m) AS map_min FROM (
    SELECT patientunitstayid, systemicmean AS m FROM `physionet-data.eicu_crd.vitalperiodic`
    WHERE observationoffset BETWEEN 0 AND 1440 AND systemicmean BETWEEN 20 AND 200
    UNION ALL
    SELECT patientunitstayid, noninvasivemean FROM `physionet-data.eicu_crd.vitalaperiodic`
    WHERE observationoffset BETWEEN 0 AND 1440 AND noninvasivemean BETWEEN 20 AND 200
  ) GROUP BY 1
),
meds AS (
  SELECT patientunitstayid,
    MAX(IF(REGEXP_CONTAINS(d, r'vancomycin') AND NOT REGEXP_CONTAINS(r, r'po|oral|enteral|tube'), 1, 0)) AS vanco,
    MAX(IF(REGEXP_CONTAINS(d, r'piperacillin|zosyn'), 1, 0)) AS piptazo,
    MAX(IF(REGEXP_CONTAINS(d, r'furosemide|lasix|bumetanide|bumex|torsemide'), 1, 0)) AS loop,
    MAX(IF(REGEXP_CONTAINS(d, r'bicarbonate'), 1, 0)) AS bicarb_med
  FROM (
    SELECT patientunitstayid, LOWER(drugname) d, LOWER(IFNULL(routeadmin, '')) r
    FROM `physionet-data.eicu_crd.medication`
    WHERE drugstartoffset BETWEEN -1440 AND 2880 AND IFNULL(drugordercancelled, 'No') = 'No'
    UNION ALL
    SELECT patientunitstayid, LOWER(drugname), 'iv'
    FROM `physionet-data.eicu_crd.infusiondrug`
    WHERE infusionoffset BETWEEN -360 AND 2880
  ) GROUP BY 1
),
rbc_tx AS (
  SELECT DISTINCT patientunitstayid FROM `physionet-data.eicu_crd.treatment`
  WHERE REGEXP_CONTAINS(LOWER(treatmentstring), r'packed red blood cells|red blood cell') AND treatmentoffset BETWEEN -360 AND 2880
),
dx AS (
  SELECT patientunitstayid,
    MAX(IF(REGEXP_CONTAINS(c, r'\b585\.|\bN18\.') OR REGEXP_CONTAINS(s, r'chronic kidney disease|chronic renal'), 1, 0)) AS ckd,
    MAX(IF(REGEXP_CONTAINS(c, r'\b250\.|\bE1[0-4]\.') OR REGEXP_CONTAINS(s, r'diabetes'), 1, 0)) AS diabetes,
    MAX(IF(REGEXP_CONTAINS(c, r'\b428\.|\bI50\.') OR REGEXP_CONTAINS(s, r'congestive heart failure|\bchf\b|heart failure'), 1, 0)) AS hf,
    MAX(IF(REGEXP_CONTAINS(c, r'571\.[256]|K70\.3|K71\.7|K74\.[3-6]') OR REGEXP_CONTAINS(s, r'cirrhosis'), 1, 0)) AS cirrhosis,
    MAX(IF(REGEXP_CONTAINS(c, r'\b038\.|995\.9[12]|785\.52|\bA4[01]\.|R65\.2') OR REGEXP_CONTAINS(s, r'sepsis|septic'), 1, 0)) AS sepsis
  FROM (
    SELECT patientunitstayid, IFNULL(icd9code, '') c, LOWER(diagnosisstring) s
    FROM `physionet-data.eicu_crd.diagnosis`
    UNION ALL
    SELECT patientunitstayid, '', LOWER(pasthistorypath) FROM `physionet-data.eicu_crd.pasthistory`
  ) GROUP BY 1
)
SELECT e.patientunitstayid,
  k.krt_start,
  IF(v.patientunitstayid IS NULL, 0, 1) AS vaso,
  IF(m.patientunitstayid IS NULL, 0, 1) AS mv,
  io.intake_ml, io.output_ml, io.urine_ml, io.n_urine_rec,
  GREATEST(IFNULL(io.rbc_io, 0), IF(r.patientunitstayid IS NULL, 0, 1)) AS rbc,
  mp.map_min,
  IFNULL(md.vanco, 0) vanco, IFNULL(md.piptazo, 0) piptazo, IFNULL(md.loop, 0) loop, IFNULL(md.bicarb_med, 0) bicarb,
  IFNULL(dx.ckd, 0) ckd, IFNULL(dx.diabetes, 0) diabetes, IFNULL(dx.hf, 0) hf,
  IFNULL(dx.cirrhosis, 0) cirrhosis, IFNULL(dx.sepsis, 0) sepsis
FROM elig e
LEFT JOIN krt k USING (patientunitstayid)
LEFT JOIN vaso v USING (patientunitstayid)
LEFT JOIN mv m USING (patientunitstayid)
LEFT JOIN io USING (patientunitstayid)
LEFT JOIN rbc_tx r USING (patientunitstayid)
LEFT JOIN map mp USING (patientunitstayid)
LEFT JOIN meds md USING (patientunitstayid)
LEFT JOIN dx USING (patientunitstayid)
""")
print("tx rows:", len(tx))


In [ ]:
# Cell 5: build the analysis dataset
def build(stay, labs, tx):
    L = labs.copy()
    L["t"] = L["t"].astype(float); L["v"] = L["v"].astype(float)

    # --- calculated standard base excess (Van Slyke) from pH and HCO3 reported at the same time
    ph = L[(L.nm == "ph") & L.v.between(6.5, 8.0)].groupby(["patientunitstayid", "t"]).v.mean().rename("ph")
    hc = L[(L.nm == "hco3") & L.v.between(2, 60)].groupby(["patientunitstayid", "t"]).v.mean().rename("hco3")
    gas = pd.concat([ph, hc], axis=1, join="inner").reset_index()
    gas["sbe"] = 0.9287 * (gas.hco3 - 24.4 + 14.83 * (gas.ph - 7.4))
    gas = gas[gas.t.between(-360, 2880)].copy()
    gas["day"] = np.where(gas.t <= 1440, 1, 2)
    lac = L[(L.nm == "lactate") & (L.v > 0) & (L.v < 30) & L.t.between(-360, 2880)].copy()
    lac["day"] = np.where(lac.t <= 1440, 1, 2)

    # primary: daily mean SBE + daily mean lactate
    ms = gas.groupby(["patientunitstayid", "day"]).sbe.mean().unstack()
    ml = lac.groupby(["patientunitstayid", "day"]).v.mean().unstack()
    nb = gas.groupby(["patientunitstayid", "day"]).size().unstack()
    D = pd.DataFrame(index=stay.patientunitstayid.values)
    D.index.name = "patientunitstayid"
    D["sbe1"] = ms.get(1); D["sbe2"] = ms.get(2)
    D["lac1"] = ml.get(1); D["lac2"] = ml.get(2)
    D["ngas1"] = nb.get(1); D["ngas2"] = nb.get(2)
    D["abe1"] = D.sbe1 + D.lac1
    D["abe2"] = D.sbe2 + D.lac2
    D["dabe"] = D.abe2 - D.abe1
    D["dbe"] = D.sbe2 - D.sbe1
    D["dlac"] = D.lac2 - D.lac1
    D["abe1_min"] = (gas.groupby(["patientunitstayid", "day"]).sbe.min().unstack().get(1)) + D.lac1

    # sensitivity: SBE-lactate pairs within 60 min (nearest lactate), daily mean of paired ABE
    g = gas[["patientunitstayid", "t", "day", "sbe"]].sort_values("t")
    l2 = lac[["patientunitstayid", "t", "v"]].rename(columns={"t": "tl"}).sort_values("tl")
    pm = pd.merge_asof(g, l2, left_on="t", right_on="tl", by="patientunitstayid",
                       direction="nearest", tolerance=60).dropna(subset=["v"])
    pm["abe"] = pm.sbe + pm.v
    pa = pm.groupby(["patientunitstayid", "day"]).abe.mean().unstack()
    D["abe1_p60"] = pa.get(1); D["abe2_p60"] = pa.get(2)
    D["dabe_p60"] = D.abe2_p60 - D.abe1_p60

    # --- creatinine and KDIGO with rolling references (lowest value in the preceding 48 h and 7 days)
    cr = L[(L.nm == "creatinine") & (L.v > 0) & (L.v < 30)][["patientunitstayid", "t", "v"]]
    cr = cr.groupby(["patientunitstayid", "t"]).v.mean().reset_index().sort_values(["patientunitstayid", "t"])
    rows = []
    for sid, gdf in cr.groupby("patientunitstayid"):
        t = gdf.t.values; v = gdf.v.values
        for i in range(len(t)):
            prev48 = v[(t >= t[i] - 2880) & (t < t[i])]
            prev7 = v[(t >= t[i] - 10080) & (t < t[i])]
            r48 = prev48.min() if len(prev48) else np.nan
            r7 = prev7.min() if len(prev7) else np.nan
            s = 0
            if not np.isnan(r7):
                if v[i] >= 3.0 * r7 or (v[i] >= 4.0 and ((not np.isnan(r48) and r48 <= 3.7) or v[i] >= 1.5 * r7)):
                    s = 3
                elif v[i] >= 2.0 * r7:
                    s = 2
                elif v[i] >= 1.5 * r7 or (not np.isnan(r48) and v[i] >= r48 + 0.3):
                    s = 1
            elif not np.isnan(r48) and v[i] >= r48 + 0.3:
                s = 1
            rows.append((sid, t[i], v[i], s))
    K = pd.DataFrame(rows, columns=["patientunitstayid", "t", "v", "stage"])

    pre = cr[cr.t < 0].groupby("patientunitstayid").v.min()
    adm = cr[cr.t.between(-720, 360)].sort_values("t").groupby("patientunitstayid").v.first()
    D["cr_adm_avail"] = D.index.isin(adm.index).astype(int)
    D["base_cr"] = pre.reindex(D.index).fillna(adm.reindex(D.index))
    D["base_pre"] = D.index.isin(pre.index).astype(int)
    D["cr_max48"] = cr[cr.t.between(-720, 2880)].groupby("patientunitstayid").v.max()
    D["cr_ratio"] = D.cr_max48 / D.base_cr

    early = K[K.t.between(-720, 2880)]
    D["early_s23"] = early[early.stage >= 2].groupby("patientunitstayid").size().reindex(D.index).fillna(0).gt(0).astype(int)
    D["stage1_48"] = early[early.stage >= 1].groupby("patientunitstayid").size().reindex(D.index).fillna(0).gt(0).astype(int)

    S = stay.set_index("patientunitstayid")
    T = tx.set_index("patientunitstayid")
    D = D.join(S).join(T)
    death_t = np.where(D.unitdischargestatus == "Expired", D.unitdischargeoffset,
              np.where(D.hospitaldischargestatus == "Expired", D.hospitaldischargeoffset, np.nan))
    D["death_t"] = death_t
    D["hosp_death"] = (D.hospitaldischargestatus == "Expired").astype(int)

    late = K[(K.t > 2880) & (K.t <= 10080)].merge(D[["death_t"]], left_on="patientunitstayid", right_index=True)
    late = late[late.death_t.isna() | (late.t <= late.death_t)]
    aki_t = late[late.stage >= 2].groupby("patientunitstayid").t.min()
    aki3_t = late[late.stage >= 3].groupby("patientunitstayid").t.min()
    krt_late = D.krt_start.where((D.krt_start > 2880) & (D.krt_start <= 10080) &
                                 (D.death_t.isna() | (D.krt_start <= D.death_t)))
    D["outcome"] = (D.index.isin(aki_t.index) | krt_late.notna()).astype(int)
    D["outcome_s3"] = (D.index.isin(aki3_t.index) | krt_late.notna()).astype(int)
    D["has_cr_after48"] = D.index.isin(cr[(cr.t > 2880) & (cr.t <= 10080)].patientunitstayid.unique()).astype(int)
    D["death7_noevent"] = ((D.death_t > 2880) & (D.death_t <= 10080) & (D.outcome == 0)).astype(int)

    # --- covariates
    def labagg(name, lo, hi, fn, t0=-360, t1=1440):
        x = L[(L.nm == name) & L.v.between(lo, hi) & L.t.between(t0, t1)]
        return x.groupby("patientunitstayid").v.agg(fn)
    D["plt_min"] = labagg("platelets x 1000", 1, 2000, "min")
    D["bili_max"] = labagg("total bilirubin", 0.05, 70, "max")
    D["bun_max"] = labagg("bun", 1, 300, "max")
    D["hgb_min"] = labagg("hgb", 2, 25, "min")
    D["cl1"] = labagg("chloride", 60, 160, "mean", -360, 1440)
    D["cl2"] = labagg("chloride", 60, 160, "mean", 1441, 2880)
    D["dcl"] = D.cl2 - D.cl1
    D["na1"] = labagg("sodium", 100, 190, "mean", -360, 1440)
    D["na2"] = labagg("sodium", 100, 190, "mean", 1441, 2880)
    D["dna"] = D.na2 - D.na1
    D["alb"] = labagg("albumin", 0.5, 7, "min", -360, 2880)
    D["male"] = (D.gender == "Male").astype(int)
    D["weight"] = D.admissionweight.where(D.admissionweight.between(30, 300))
    D["fb_l"] = (D.intake_ml.fillna(0) - D.output_ml.fillna(0)) / 1000
    D.loc[D.intake_ml.isna() & D.output_ml.isna(), "fb_l"] = np.nan
    lo_, hi_ = D.fb_l.quantile([0.01, 0.99]); D["fb_l"] = D.fb_l.clip(lo_, hi_)
    D["uo"] = (D.urine_ml / D.weight / 48).where(D.n_urine_rec > 0).clip(upper=5)
    icu_map = {"MICU": "Medical", "SICU": "Surgical", "Med-Surg ICU": "Med-Surg",
               "CCU-CTICU": "Coronary/cardiac", "Cardiac ICU": "Coronary/cardiac",
               "CSICU": "Cardiothoracic", "CTICU": "Cardiothoracic", "Neuro ICU": "Neuro"}
    D["icu"] = D.unittype.map(icu_map).fillna("Other")
    src = D.unitadmitsource.fillna("Other")
    D["adm_src"] = np.select(
        [src.str.contains("Emergency"), src.str.contains("Operating|Recovery|PACU"),
         src.str.contains("Floor|Step-Down|Telemetry|ICU|Acute Care")],
        ["Emergency", "Operating room", "Ward/other ICU"], "Other")
    D["elective"] = D.electivesurgery.fillna(0).astype(int)
    D["dabe_cat"] = pd.cut(D.dabe, [-np.inf, -2, 0, 2, np.inf], labels=["<=-2", "-2-0", "0-2", ">2"])

    # --- cohort flow
    flow = {"eligible blood gas/lactate stays (either definition)": len(D)}
    D = D[D.cr_adm_avail == 1]; flow["admission creatinine available"] = len(D)
    D = D[~((D.krt_start.notna()) & (D.krt_start <= 2880))]; flow["no KRT within 48 h"] = len(D)
    D = D[D.early_s23 == 0]; flow["no KDIGO stage 2-3 within 48 h (final, either definition)"] = len(D)
    flow["primary definition (daily mean)"] = int(D.dabe.notna().sum())
    flow["sensitivity definition (pairs within 60 min)"] = int(D.dabe_p60.notna().sum())
    return D, flow

D_all, flow = build(stay, labs, tx)
D = D_all[D_all.dabe.notna()].copy()   # primary eICU analysis cohort


In [ ]:
# Cell 6: aggregated summary to share (no patient-level data)
import statsmodels.formula.api as smf
pd.set_option("display.width", 200)
print("=== FLOW ==="); [print(f"{k}: {v}") for k, v in flow.items()]
print("\n=== PRIMARY COHORT ===")
print("n =", len(D), "| events =", int(D.outcome.sum()), f"({D.outcome.mean()*100:.1f}%)",
      "| stage 3/KRT events =", int(D.outcome_s3.sum()),
      "| hospitals =", D.hospitalid.nunique())
print("KRT events (48 h-day 7):", int(((D.krt_start > 2880) & (D.krt_start <= 10080)).sum()))
print("creatinine after 48 h available: %.1f%%" % (D.has_cr_after48.mean() * 100))
print("stage 1 AKI by 48 h: %.1f%% (outcome) vs %.1f%% (no outcome)" % (
      D[D.outcome == 1].stage1_48.mean() * 100, D[D.outcome == 0].stage1_48.mean() * 100))
print("death 48 h-day 7 without outcome:", int(D.death7_noevent.sum()),
      "| in-hospital mortality: %.1f%%" % (D.hosp_death.mean() * 100))
print("baseline creatinine from pre-ICU value: %.1f%%" % (D.base_pre.mean() * 100))

cont = ["age", "abe1", "abe2", "dabe", "sbe1", "lac1", "dlac", "dbe", "base_cr", "cr_ratio", "aps",
        "map_min", "fb_l", "uo", "plt_min", "bili_max", "bun_max", "hgb_min", "cl1", "dcl", "ngas1", "ngas2"]
binv = ["male", "vaso", "mv", "vanco", "piptazo", "loop", "bicarb", "rbc", "ckd", "diabetes", "hf",
        "cirrhosis", "sepsis", "elective", "stage1_48"]
def miqr(x): return f"{x.median():.2f} ({x.quantile(.25):.2f} to {x.quantile(.75):.2f})"
print("\n=== CONTINUOUS: median (IQR) overall | no outcome | outcome | % missing ===")
for c in cont:
    print(f"{c:10s} {miqr(D[c])} | {miqr(D[D.outcome==0][c])} | {miqr(D[D.outcome==1][c])} | {D[c].isna().mean()*100:.1f}%")
print("\n=== BINARY: % overall | no outcome | outcome ===")
for c in binv:
    print(f"{c:10s} {D[c].mean()*100:.1f} | {D[D.outcome==0][c].mean()*100:.1f} | {D[D.outcome==1][c].mean()*100:.1f}")
print("\nICU type:\n", pd.crosstab(D.icu, D.outcome, margins=True))
print("\nAdmission source:\n", pd.crosstab(D.adm_src, D.outcome, margins=True))
print("\n=== EVENT RATE BY ΔABE CATEGORY ===")
print(D.groupby("dabe_cat", observed=True).outcome.agg(["size", "sum", "mean"]))
m0 = smf.logit("outcome ~ dabe", D).fit(disp=0)
m1 = smf.logit("outcome ~ dabe + abe1", D).fit(disp=0)
for nm_, m in [("crude", m0), ("+ day-1 ABE", m1)]:
    print(f"OR per 1 mmol/L ΔABE ({nm_}): {np.exp(m.params['dabe']):.3f} "
          f"({np.exp(m.conf_int().loc['dabe', 0]):.3f}-{np.exp(m.conf_int().loc['dabe', 1]):.3f}), P = {m.pvalues['dabe']:.4f}")
Dp = D_all[D_all.dabe_p60.notna()]
mp_ = smf.logit("outcome ~ dabe_p60 + abe1_p60", Dp).fit(disp=0)
print(f"Pairs within 60 min: n = {len(Dp)}, events = {int(Dp.outcome.sum())}, OR = {np.exp(mp_.params['dabe_p60']):.3f} "
      f"({np.exp(mp_.conf_int().loc['dabe_p60', 0]):.3f}-{np.exp(mp_.conf_int().loc['dabe_p60', 1]):.3f})")


Stage 2: analysis (run after Stage 1 cells; uses D and D_all)
Prints only aggregated results.

In [ ]:
# Cell 7: imputation and pooling helpers
import warnings; warnings.filterwarnings("ignore")
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm
from scipy import stats

def prep(df):
    X = pd.DataFrame(index=df.index)
    X["dabe"] = df.dabe; X["abe1"] = df.abe1
    X["age"] = df.age; X["male"] = df.male
    for lvl in ["Operating room", "Ward/other ICU", "Other"]:
        X["src_" + lvl] = (df.adm_src == lvl).astype(int)          # ref: Emergency
    for lvl in ["Medical", "Surgical", "Coronary/cardiac", "Cardiothoracic", "Neuro", "Other"]:
        X["icu_" + lvl] = (df.icu == lvl).astype(int)              # ref: Med-Surg
    X["elective"] = df.elective
    X["log_cr"] = np.log(df.base_cr); X["base_pre"] = df.base_pre
    X["log_lac1"] = np.log(df.lac1)
    for c in ["vaso", "mv", "fb_l", "uo", "map_min", "plt_min", "hgb_min",
              "vanco", "piptazo", "loop", "bicarb", "rbc", "ckd", "diabetes", "hf", "cirrhosis", "sepsis"]:
        X[c] = df[c]
    X["log_bili"] = np.log(df.bili_max); X["log_bun"] = np.log(df.bun_max)
    X = X.loc[:, X.nunique(dropna=True) > 1]   # drop constant columns (restricted cohorts)
    aux = pd.DataFrame({"aps": df.aps, "log_cr_ratio": np.log(df.cr_ratio), "dcl": df.dcl,
                        "outcome": df.outcome, "dabe_p60": df.get("dabe_p60")}, index=df.index)
    return X, aux

COV = None
def impute(df, m=40, seed=2026):
    X, aux = prep(df)
    full = pd.concat([X, aux.drop(columns=["dabe_p60"])], axis=1)
    out = []
    for i in range(m):
        imp = IterativeImputer(estimator=BayesianRidge(), sample_posterior=True, max_iter=20, random_state=seed + i)
        arr = imp.fit_transform(full)
        d = pd.DataFrame(arr, columns=full.columns, index=full.index)
        for c in full.columns:            # keep observed values exactly
            d.loc[full[c].notna(), c] = full.loc[full[c].notna(), c]
        out.append(d)
    return out

def rubin(est, var):
    est = np.asarray(est); var = np.asarray(var); m = len(est)
    qbar = est.mean(0); ubar = var.mean(0); b = est.var(0, ddof=1)
    t = ubar + (1 + 1 / m) * b
    r = (1 + 1 / m) * b / np.where(ubar > 0, ubar, np.nan)
    dfree = (m - 1) * (1 + 1 / np.where(r > 0, r, np.nan)) ** 2
    return qbar, np.sqrt(t), dfree

def fit_pool(imps, y, cols, term="dabe", groups=None, df_src=None):
    est, var = [], []
    cols = [c for c in cols if imps[0][c].nunique() > 1]
    for d in imps:
        Xd = sm.add_constant(d[cols], has_constant="add")
        if groups is not None:
            r = sm.Logit(y, Xd).fit(disp=0, cov_type="cluster", cov_kwds={"groups": groups})
        else:
            r = sm.Logit(y, Xd).fit(disp=0)
        est.append(r.params[term]); var.append(r.bse[term] ** 2)
    q, se, dfree = rubin(est, var)
    dfree = np.nan_to_num(dfree, nan=1e6)
    tcrit = stats.t.ppf(0.975, dfree)
    p = 2 * stats.t.sf(abs(q / se), dfree)
    return np.exp(q), np.exp(q - tcrit * se), np.exp(q + tcrit * se), p

def fmt(r): return f"{r[0]:.3f} ({r[1]:.3f}-{r[2]:.3f}), P = {r[3]:.4f}"

# fast DeLong for paired AUROC difference
def _midrank(x):
    j = np.argsort(x); z = x[j]; n = len(x); t = np.zeros(n); i = 0
    while i < n:
        k = i
        while k < n and z[k] == z[i]: k += 1
        t[i:k] = 0.5 * (i + k - 1) + 1; i = k
    out = np.empty(n); out[j] = t; return out
def delong(y, p1, p2):
    y = np.asarray(y); pos = y == 1; neg = ~pos; m_, n_ = pos.sum(), neg.sum()
    aucs, v10, v01 = [], [], []
    for p in (p1, p2):
        p = np.asarray(p); tx = _midrank(p[pos]); ty = _midrank(p[neg]); tz = _midrank(p)
        auc = (tz[pos].sum() - m_ * (m_ + 1) / 2) / (m_ * n_)
        aucs.append(auc); v10.append((tz[pos] - tx) / n_); v01.append(1 - (tz[neg] - ty) / m_)
    s10 = np.cov(np.vstack(v10)); s01 = np.cov(np.vstack(v01))
    var = (s10[0, 0] + s10[1, 1] - 2 * s10[0, 1]) / m_ + (s01[0, 0] + s01[1, 1] - 2 * s01[0, 1]) / n_
    return aucs[0], aucs[1], aucs[1] - aucs[0], var

def auc_compare(imps, y, cols_a, cols_b):
    a1, a2, dif, var = [], [], [], []
    cols_a = [c for c in cols_a if imps[0][c].nunique() > 1]
    cols_b = [c for c in cols_b if imps[0][c].nunique() > 1]
    for d in imps:
        pa = sm.Logit(y, sm.add_constant(d[cols_a], has_constant="add")).fit(disp=0).predict()
        pb = sm.Logit(y, sm.add_constant(d[cols_b], has_constant="add")).fit(disp=0).predict()
        r = delong(y, pa, pb); a1.append(r[0]); a2.append(r[1]); dif.append(r[2]); var.append(r[3])
    q, se, dfree = rubin(dif, var)
    p = 2 * stats.norm.sf(abs(q / se))
    return np.mean(a1), np.mean(a2), q, q - 1.96 * se, q + 1.96 * se, p


In [ ]:
# Cell 8: primary analyses
np.random.seed(2026)
y = D.outcome.values
grp = D.hospitalid.values
Xc, _ = prep(D)
inc = Xc.isna().any(axis=1).mean() * 100
M = max(40, int(np.ceil(inc)))
print(f"Incomplete cases: {inc:.1f}% -> {M} imputations")
imps = impute(D, m=M)
COV = [c for c in Xc.columns if c not in ("dabe",)]
FULL = ["dabe"] + COV

print("\n=== PRIMARY: OR per 1 mmol/L increase in ΔABE (eICU) ===")
print("Unadjusted:                ", fmt(fit_pool(imps, y, ["dabe"], groups=grp)))
print("+ day-1 ABE:               ", fmt(fit_pool(imps, y, ["dabe", "abe1"], groups=grp)))
res_full = fit_pool(imps, y, FULL, groups=grp)
print("Full model (cluster-robust):", fmt(res_full))
print("Full model (model-based SE):", fmt(fit_pool(imps, y, FULL)))
print("Per 1 mmol/L decrease:      ", f"{1/res_full[0]:.3f} ({1/res_full[2]:.3f}-{1/res_full[1]:.3f})")
PARS = ["dabe", "abe1", "age", "male", "log_cr", "log_lac1", "aps"]
print("Parsimonious (+APACHE APS): ", fmt(fit_pool(imps, y, PARS, groups=grp)))
print("Full + APACHE APS:          ", fmt(fit_pool(imps, y, FULL + ["aps"], groups=grp)))
print("Full + log creatinine ratio:", fmt(fit_pool(imps, y, FULL + ["log_cr_ratio"], groups=grp)))
print("Full + day-1 chloride chg:  ", fmt(fit_pool(imps, y, FULL + ["dcl"], groups=grp)))
NOFLUID = [c for c in FULL if c not in ("fb_l", "uo")]
print("Full without fluid balance and urine output:", fmt(fit_pool(imps, y, NOFLUID, groups=grp)))
NOMED = [c for c in FULL if c not in ("vanco", "piptazo", "loop", "bicarb", "rbc")]
print("Full without drugs/transfusion:             ", fmt(fit_pool(imps, y, NOMED, groups=grp)))

# categorical ΔABE (reference 0 to 2 mmol/L)
for d in imps:
    d["cat_le_m2"] = (D.dabe <= -2).astype(int).values
    d["cat_m2_0"] = ((D.dabe > -2) & (D.dabe <= 0)).astype(int).values
    d["cat_gt2"] = (D.dabe > 2).astype(int).values
CATS = ["cat_le_m2", "cat_m2_0", "cat_gt2"] + COV
print("\n=== CATEGORICAL ΔABE (ref: >0 to ≤2) ===")
for t, lab in [("cat_le_m2", "≤ -2"), ("cat_m2_0", "-2 to 0"), ("cat_gt2", "> 2")]:
    print(f"ΔABE {lab:8s}", fmt(fit_pool(imps, y, CATS, term=t, groups=grp)))


In [ ]:
# Cell 9: new-onset AKI vs progression, secondary outcome, discrimination
print("\n=== EARLY STAGE 1 AKI STRATA ===")
for s in (0, 1):
    idx = np.where(D.stage1_48.values == s)[0]
    if len(idx) == 0 or y[idx].sum() == 0: print(f"stage 1 by 48 h = {s}: no patients/events"); continue
    sub = [d.iloc[idx] for d in imps]
    cols = [c for c in FULL if sub[0][c].nunique() > 1]
    r = fit_pool(sub, y[idx], cols, groups=grp[idx])
    print(f"stage 1 by 48 h = {s}: n = {len(idx)}, events = {int(y[idx].sum())}, OR = {fmt(r)}")
for d in imps:
    d["s1"] = D.stage1_48.values; d["dabe_x_s1"] = d.dabe * d.s1
if D.stage1_48.nunique() > 1:
    r = fit_pool(imps, y, FULL + ["s1", "dabe_x_s1"], term="dabe_x_s1", groups=grp)
    print(f"P for interaction: {r[3]:.3f}")

y3 = D.outcome_s3.values
print("\nSecondary outcome (stage 3 or KRT): events =", int(y3.sum()), "| OR =", fmt(fit_pool(imps, y3, FULL, groups=grp)))

print("\n=== DISCRIMINATION (mean AUROC across imputations; paired DeLong, Rubin) ===")
a = auc_compare(imps, y, COV, FULL)
print(f"Covariates + day-1 ABE -> + ΔABE: {a[0]:.3f} -> {a[1]:.3f}; diff {a[2]:.4f} ({a[3]:.4f} to {a[4]:.4f}), P = {a[5]:.3f}")
a = auc_compare(imps, y, COV + ["log_cr_ratio"], FULL + ["log_cr_ratio"])
print(f"Creatinine model -> + ΔABE:       {a[0]:.3f} -> {a[1]:.3f}; diff {a[2]:.4f} ({a[3]:.4f} to {a[4]:.4f}), P = {a[5]:.3f}")


In [ ]:
# Cell 10: sensitivity analyses
print("\n=== SENSITIVITY ===")
# complete case
cc = Xc.dropna().index
Xcc = sm.add_constant(Xc.loc[cc, FULL], has_constant="add")
rcc = sm.Logit(D.loc[cc, "outcome"], Xcc).fit(disp=0, cov_type="cluster", cov_kwds={"groups": D.loc[cc, "hospitalid"]})
ci = np.exp(rcc.conf_int().loc["dabe"])
print(f"Complete case: n = {len(cc)}, events = {int(D.loc[cc,'outcome'].sum())}, OR = {np.exp(rcc.params['dabe']):.3f} ({ci[0]:.3f}-{ci[1]:.3f}), P = {rcc.pvalues['dabe']:.4f}")

# hospitals contributing medication data
hosp_med = D.groupby("hospitalid")[["vanco", "piptazo", "loop"]].max().max(axis=1)
keep = D.hospitalid.map(hosp_med).eq(1).values
idx = np.where(keep)[0]; sub = [d.iloc[idx] for d in imps]
print(f"Hospitals with medication data: {int(hosp_med.sum())}/{len(hosp_med)}; n = {len(idx)}, events = {int(y[idx].sum())}, OR =",
      fmt(fit_pool(sub, y[idx], [c for c in FULL if sub[0][c].nunique() > 1], groups=grp[idx])))

# pairs within 60 min (sensitivity exposure definition)
Dp = D_all[D_all.dabe_p60.notna()].copy()
Dp["dabe"] = Dp.dabe_p60; Dp["abe1"] = Dp.abe1_p60
imps_p = impute(Dp, m=20)
cols_p = [c for c in FULL if c in imps_p[0].columns and imps_p[0][c].nunique() > 1]
print(f"SBE-lactate pairs within 60 min: n = {len(Dp)}, events = {int(Dp.outcome.sum())}, OR =",
      fmt(fit_pool(imps_p, Dp.outcome.values, cols_p, groups=Dp.hospitalid.values)))

# exclude cardiothoracic ICU (cardiac surgery-like population)
idx = np.where(D.icu.values != "Cardiothoracic")[0]; sub = [d.iloc[idx] for d in imps]
print(f"Excluding cardiothoracic ICU: n = {len(idx)}, events = {int(y[idx].sum())}, OR =",
      fmt(fit_pool(sub, y[idx], [c for c in FULL if sub[0][c].nunique() > 1], groups=grp[idx])))

# components: ΔBE (calculated SBE) and Δlactate
for d in imps:
    d["dbe"] = D.dbe.values; d["sbe1"] = D.sbe1.values; d["dlac"] = D.dlac.values
COV_BE = [c for c in COV if c != "abe1"] + ["sbe1"]
print("ΔSBE (with day-1 SBE):      ", fmt(fit_pool(imps, y, ["dbe"] + COV_BE, term="dbe", groups=grp)))
print("ΔLactate (per 1 mmol/L):    ", fmt(fit_pool(imps, y, ["dlac"] + [c for c in COV if c != "abe1"], term="dlac", groups=grp)))
print("ΔABE at a given ΔLactate:   ", fmt(fit_pool(imps, y, FULL + ["dlac"], groups=grp)))
print("\nDone.")
